# 01.5 Building Models with `nn.Module`

This notebook starts working with actual models. `nn.Module` is the core abstraction behind PyTorch models: it stores layers, exposes parameters, defines the forward pass, and lets submodules be composed into larger architectures.

The main ideas are module structure, layers, parameters, submodules, activation functions, and how input shapes become output shapes through a model.

## Learning Goals

After this notebook, you should be able to:

1. Understand the basic structure of `nn.Module`.
2. Define simple model classes yourself.
3. Distinguish layers from parameters.
4. Read what `forward` is doing.
5. Build a simple MLP.
6. Prepare the model component for later training loops.

In [ ]:
import torch
import torch.nn as nn

## The Minimal Structure of `nn.Module`

A minimal model usually has two parts:

1. define layers
2. define how data flows through the layers

In [ ]:
class SimpleLinearModel(nn.Module):
    def __init__(self, in_features, out_features):
        super().__init__()
        self.linear = nn.Linear(in_features, out_features)

    def forward(self, x):
        return self.linear(x)


model = SimpleLinearModel(in_features=2, out_features=1)
print(model)

The meaning of this first model is direct. Each input sample has 2 features, and the layer produces 1 output value. When you call `model(x)`, PyTorch internally calls the model's `forward` method and passes `x` through the linear layer.

The important habit is to track both the sample dimension and the feature dimension. If `x.shape == (5, 2)`, there are 5 samples and each sample has 2 features, so `nn.Linear(2, 1)` can process it.

## Forward Pass

`forward` defines how inputs become outputs.

In `PyTorch`, you usually do not write `model.forward(x)` directly. You write `model(x)`.


In [ ]:
x = torch.tensor([[1.0, 2.0], [3.0, 4.0]])
out = model(x)

print("x.shape =", x.shape)
print("out =\n", out)
print("out.shape =", out.shape)

Because `out_features=1`, the output shape is `(batch_size, 1)`.


In [ ]:
# Exercise 1
#
# Create a SimpleLinearModel with input dimension 3 and output dimension 2.
#
# Then:
# - Create x_ex with shape (4, 3), meaning 4 samples and 3 features per sample.
# - Pass x_ex through the model.
# - Print out_ex.shape.
#
# Expected output shape:
# - (4, 2), because the batch size stays 4 and the layer outputs 2 values per sample.

# model_ex =
# x_ex =
# out_ex =
# print(out_ex.shape)

In [ ]:
# Exercise 1 Reference Solution

model_ex = SimpleLinearModel(in_features=3, out_features=2)
x_ex = torch.randn(4, 3)
out_ex = model_ex(x_ex)
print(out_ex.shape)

## Parameters

A model can learn because it contains parameters. In a linear layer, the weight matrix controls how input features are combined, and the bias shifts the output. During training, gradients are computed for these parameters and the optimizer updates them.

When you inspect model parameters, you are inspecting the values that training will change.

In [ ]:
for name, param in model.named_parameters():
    print(name)
    print("shape =", param.shape)
    print(param)
    print()

When you later see `model.parameters()` or `model.named_parameters()`, you are essentially looking at the learnable parts of the model.


## Adding an Activation Function

With only linear layers, the model has limited expressive power because stacking linear transformations without nonlinearity still behaves like one larger linear transformation. Activation functions such as ReLU, Sigmoid, and Tanh introduce nonlinearity, which lets the network represent more complex relationships.

In [ ]:
class TwoLayerMLP(nn.Module):
    def __init__(self, in_features, hidden_features, out_features):
        super().__init__()
        self.fc1 = nn.Linear(in_features, hidden_features)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_features, out_features)

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        return x


mlp = TwoLayerMLP(in_features=2, hidden_features=4, out_features=1)
print(mlp)

In [ ]:
x = torch.randn(5, 2)
out = mlp(x)
print("x.shape =", x.shape)
print("out.shape =", out.shape)
print(out)

The shape flow is the part to watch. The input starts as `(5, 2)`, meaning 5 samples with 2 features each. The first layer maps each sample to 4 hidden values, giving `(5, 4)`. The second layer maps those hidden values to 1 output value per sample, giving `(5, 1)`.

In [ ]:
# Exercise 2
#
# Define a two-layer MLP.
#
# Use the existing TwoLayerMLP class with:
# - input dimension 4
# - hidden dimension 8
# - output dimension 3
#
# Then:
# - Create x2 with shape (6, 4).
# - Pass x2 through the model.
# - Print out2.shape.
#
# Expected shape:
# - (6, 3), because there are 6 samples and the model outputs 3 values per sample.

# model2 =
# x2 =
# out2 =
# print(out2.shape)

In [ ]:
# Exercise 2 Reference Solution

model2 = TwoLayerMLP(in_features=4, hidden_features=8, out_features=3)
x2 = torch.randn(6, 4)
out2 = model2(x2)
print(out2.shape)

## 5. `nn.Sequential`

If the model is a simple chain of layers, `nn.Sequential` can make the code more compact.


In [ ]:
seq_model = nn.Sequential(
    nn.Linear(2, 4),
    nn.ReLU(),
    nn.Linear(4, 1),
)

print(seq_model)

x = torch.randn(3, 2)
out = seq_model(x)
print("out.shape =", out.shape)

`nn.Sequential` is great for simple chains, but when you need branches, skip connections, or multiple inputs/outputs, a custom `forward` is usually clearer.


## Output Layer and Task Type

The final layer must match the meaning of the task. Regression usually outputs one or more continuous values. Binary classification can output one logit with `BCEWithLogitsLoss` or two class logits with `CrossEntropyLoss`. Multiclass classification usually outputs one score per class.

This is why output shape and loss function should be chosen together. A model output can have the right-looking numbers but still be incompatible with the loss if the task setup is mismatched.

In [ ]:
reg_model = TwoLayerMLP(in_features=2, hidden_features=4, out_features=1)
cls_model = TwoLayerMLP(in_features=2, hidden_features=4, out_features=3)

x = torch.randn(4, 2)
print("regression output shape =", reg_model(x).shape)
print("classification output shape =", cls_model(x).shape)

In [ ]:
# Exercise 3
#
# For each task below, decide the usual output dimension and explain why.
#
# Tasks:
# 1. Predict house price.
# 2. Classify whether an email is spam or not spam.
#
# Write your answer in full sentences. For the spam example, mention that the
# output dimension depends on whether you use one binary logit or two class
# logits.

Exercise 3 Reference Answer

1. House price prediction usually uses output dimension `1`, because it predicts one continuous value.
2. Spam/not-spam classification can use output dimension `2` with `CrossEntropyLoss`, or output dimension `1` with `BCEWithLogitsLoss`.

The important habit is to choose the output layer together with the loss function, not in isolation.

## Summary

The key lesson here is not memorizing class names, but understanding the fixed structure of model definition:

1. define layers in `__init__`
2. define data flow in `forward`
3. let the model learn through parameters

You should now be able to answer:

1. Why is `nn.Module` the core abstraction for models?
2. Why are layers usually defined in `__init__` instead of inside `forward`?
3. Why does output dimension depend on task type?
4. When is `nn.Sequential` suitable and when is a custom `forward` better?

Suggested next step:

- Move to the loss-and-optimizer notebook to connect model outputs to learning.